# Population

This was an exceptionally complex cleaning job. 

In [1]:
pip install pandas openpyxl xlrd

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd

# Set your input and output folders
input_path = '/Users/matthewcavanaugh/Desktop/GitHub/us-presidential-elections-by-county/sources/population raw/State Pop Estimates 2000 to 2010 unconverted'
output_path = '/Users/matthewcavanaugh/Desktop/GitHub/us-presidential-elections-by-county/sources/population raw/State Pop Estimates 2000 to 2010 CSV'

# Ensure the output folder exists
os.makedirs(output_path, exist_ok=True)

# Loop through all .xls files in the input folder
for filename in os.listdir(input_path):
    if filename.endswith('.xls'):
        xls_path = os.path.join(input_path, filename)
        csv_filename = filename.replace('.xls', '.csv')
        csv_path = os.path.join(output_path, csv_filename)

        try:
            df = pd.read_excel(xls_path, engine='xlrd')
            df.to_csv(csv_path, index=False)
            print(f'Converted: {filename} -> {csv_filename}')
        except Exception as e:
            print(f'Failed to convert {filename}: {e}')

Converted: co-est00int-01-37.xls -> co-est00int-01-37.csv
Converted: co-est00int-01-23.xls -> co-est00int-01-23.csv
Converted: co-est00int-01-22.xls -> co-est00int-01-22.csv
Converted: co-est00int-01-36.xls -> co-est00int-01-36.csv
Converted: co-est00int-01-20.xls -> co-est00int-01-20.csv
Converted: co-est00int-01-34.xls -> co-est00int-01-34.csv
Converted: co-est00int-01-08.xls -> co-est00int-01-08.csv
Converted: co-est00int-01-09.xls -> co-est00int-01-09.csv
Converted: co-est00int-01-35.xls -> co-est00int-01-35.csv
Converted: co-est00int-01-21.xls -> co-est00int-01-21.csv
Converted: co-est00int-01-19.xls -> co-est00int-01-19.csv
Converted: co-est00int-01-25.xls -> co-est00int-01-25.csv
Converted: co-est00int-01-31.xls -> co-est00int-01-31.csv
Converted: co-est00int-01-30.xls -> co-est00int-01-30.csv
Converted: co-est00int-01-24.xls -> co-est00int-01-24.csv
Converted: co-est00int-01-18.xls -> co-est00int-01-18.csv
Converted: co-est00int-01-32.xls -> co-est00int-01-32.csv
Converted: co-

In [3]:
import pandas as pd
import os
from glob import glob

# List of all US states + DC
states_list = [
    'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut',
    'Delaware', 'District of Columbia', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois',
    'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts',
    'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada',
    'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota',
    'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina',
    'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington',
    'West Virginia', 'Wisconsin', 'Wyoming'
]

folder_path = '/Users/matthewcavanaugh/Desktop/GitHub/us-presidential-elections-by-county/sources/population raw/State Pop Estimates 2000 to 2010 CSV'
csv_files = glob(os.path.join(folder_path, "*.csv"))

df_list = []

for file in csv_files:
    df = pd.read_csv(file, skiprows=2, skipfooter=8, engine='python')
    df = df.dropna(how='all')
    df.columns = df.columns.str.strip().str.lower().str.replace(r'[^\w\s]', '', regex=True)

    geo_col = 'geographic area'

    # Strip whitespace to avoid false mismatches
    df[geo_col] = df[geo_col].str.strip()

    # Identify exact matches to state names
    df['state_temp'] = df[geo_col].where(df[geo_col].isin(states_list))

    # Forward-fill to propagate state name down
    df['state'] = df['state_temp'].ffill()

    # Remove rows that are state rows themselves (i.e., state summary rows)
    df = df[~df[geo_col].isin(states_list)]

    df = df.drop(columns=['state_temp'])

    df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)

# Optional: create full county name
combined_df['full_county_name'] = combined_df['geographic area'] + ", " + combined_df['state']

In [4]:
combined_df

,geographic area,april 1 20001,intercensal estimates as of july 1,unnamed 3,unnamed 4,unnamed 5,unnamed 6,unnamed 7,unnamed 8,unnamed 9,unnamed 10,unnamed 11,april 1 20102,july 1 20103,state,full_county_name
0,NaN,NaN,2000,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,NaN,NaN,NaN,NaN
1,.Anderson County,55114.0,55062,54263.0,54740.0,56068.0,56245.0,56873.0,57386.0,57870.0,57963.0,58410.0,58458.0,58452.0,Texas,".Anderson County, Texas"
2,.Andrews County,13002.0,12949,12856.0,13022.0,12976.0,13006.0,13016.0,13195.0,13513.0,14099.0,14601.0,14786.0,14833.0,Texas,".Andrews County, Texas"
3,.Angelina County,80123.0,80270,80273.0,80803.0,81510.0,82070.0,82553.0,83810.0,84518.0,84961.0,86029.0,86771.0,86953.0,Texas,".Angelina County, Texas"
4,.Aransas County,22457.0,22452,22287.0,22616.0,22843.0,23067.0,23561.0,23395.0,23172.0,23225.0,23291.0,23158.0,23151.0,Texas,".Aransas County, Texas"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3189,.Washington County,203014.0,203008,203179.0,203104.0,203859.0,204090.0,205359.0,205992.0,206259.0,207240.0,207571.0,207820.0,207875.0,Pennsylvania,".Washington County, Pennsylvania"
3190,.Wayne County,47679.0,47843,47994.0,48330.0,48938.0,49423.0,50409.0,52031.0,52341.0,51692.0,52552.0,52822.0,52895.0,Pennsylvania,".Wayne County, Pennsylvania"
3191,.Westmoreland County,369990.0,369664,368698.0,367680.0,367836.0,367281.0,366946.0,366559.0,366318.0,366058.0,365259.0,365169.0,365012.0,Pennsylvania,".Westmoreland County, Pennsylvania"
3192,.Wyoming County,28072.0,28038,28119.0,28011.0,28074.0,28195.0,28160.0,28194.0,28287.0,28294.0,28268.0,28276.0,28266.0,Pennsylvania,".Wyoming County, Pennsylvania"


In [5]:
combined_df.rename(columns={'april 1 20001': '2000'}, inplace=True)
combined_df.rename(columns={'unnamed 3': '2001'}, inplace=True)
combined_df.rename(columns={'unnamed 4': '2002'}, inplace=True)
combined_df.rename(columns={'unnamed 5': '2003'}, inplace=True)
combined_df.rename(columns={'unnamed 6': '2004'}, inplace=True)
combined_df.rename(columns={'unnamed 7': '2005'}, inplace=True)
combined_df.rename(columns={'unnamed 8': '2006'}, inplace=True)
combined_df.rename(columns={'unnamed 9': '2007'}, inplace=True)
combined_df.rename(columns={'unnamed 10': '2008'}, inplace=True)
combined_df.rename(columns={'unnamed 11': '2009'}, inplace=True)
combined_df.rename(columns={'april 1 20102': '2010'}, inplace=True)

# Drop columns
combined_df.drop('intercensal estimates as of july 1', axis=1, inplace=True)
combined_df.drop('july 1 20103', axis=1, inplace=True)

In [6]:
combined_df

,geographic area,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,state,full_county_name
0,NaN,NaN,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,NaN,NaN,NaN
1,.Anderson County,55114.0,54263.0,54740.0,56068.0,56245.0,56873.0,57386.0,57870.0,57963.0,58410.0,58458.0,Texas,".Anderson County, Texas"
2,.Andrews County,13002.0,12856.0,13022.0,12976.0,13006.0,13016.0,13195.0,13513.0,14099.0,14601.0,14786.0,Texas,".Andrews County, Texas"
3,.Angelina County,80123.0,80273.0,80803.0,81510.0,82070.0,82553.0,83810.0,84518.0,84961.0,86029.0,86771.0,Texas,".Angelina County, Texas"
4,.Aransas County,22457.0,22287.0,22616.0,22843.0,23067.0,23561.0,23395.0,23172.0,23225.0,23291.0,23158.0,Texas,".Aransas County, Texas"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3189,.Washington County,203014.0,203179.0,203104.0,203859.0,204090.0,205359.0,205992.0,206259.0,207240.0,207571.0,207820.0,Pennsylvania,".Washington County, Pennsylvania"
3190,.Wayne County,47679.0,47994.0,48330.0,48938.0,49423.0,50409.0,52031.0,52341.0,51692.0,52552.0,52822.0,Pennsylvania,".Wayne County, Pennsylvania"
3191,.Westmoreland County,369990.0,368698.0,367680.0,367836.0,367281.0,366946.0,366559.0,366318.0,366058.0,365259.0,365169.0,Pennsylvania,".Westmoreland County, Pennsylvania"
3192,.Wyoming County,28072.0,28119.0,28011.0,28074.0,28195.0,28160.0,28194.0,28287.0,28294.0,28268.0,28276.0,Pennsylvania,".Wyoming County, Pennsylvania"


In [7]:
combined_df.isnull().sum()

geographic area     51
2000                51
2001                 0
2002                 0
2003                 0
2004                 0
2005                 0
2006                 0
2007                 0
2008                 0
2009                 0
2010                51
state               51
full_county_name    51
dtype: int64

In [8]:
combined_df[combined_df.isnull().any(axis=1)]

,geographic area,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,state,full_county_name
0,NaN,NaN,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,NaN,NaN,NaN
255,NaN,NaN,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,NaN,NaN,NaN
285,NaN,NaN,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,NaN,NaN,NaN
374,NaN,NaN,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,NaN,NaN,NaN
376,NaN,NaN,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,NaN,NaN,NaN
452,NaN,NaN,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,NaN,NaN,NaN
468,NaN,NaN,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,NaN,NaN,NaN
472,NaN,NaN,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,NaN,NaN,NaN
526,NaN,NaN,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,NaN,NaN,NaN
585,NaN,NaN,2001.0,2002.0,2003.0,2004.0,2005.0,2006.0,2007.0,2008.0,2009.0,NaN,NaN,NaN


In [9]:
combined_df.dropna(inplace=True)

In [10]:
combined_df.isnull().sum()

geographic area     0
2000                0
2001                0
2002                0
2003                0
2004                0
2005                0
2006                0
2007                0
2008                0
2009                0
2010                0
state               0
full_county_name    0
dtype: int64

In [11]:
combined_df

,geographic area,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,state,full_county_name
1,.Anderson County,55114.0,54263.0,54740.0,56068.0,56245.0,56873.0,57386.0,57870.0,57963.0,58410.0,58458.0,Texas,".Anderson County, Texas"
2,.Andrews County,13002.0,12856.0,13022.0,12976.0,13006.0,13016.0,13195.0,13513.0,14099.0,14601.0,14786.0,Texas,".Andrews County, Texas"
3,.Angelina County,80123.0,80273.0,80803.0,81510.0,82070.0,82553.0,83810.0,84518.0,84961.0,86029.0,86771.0,Texas,".Angelina County, Texas"
4,.Aransas County,22457.0,22287.0,22616.0,22843.0,23067.0,23561.0,23395.0,23172.0,23225.0,23291.0,23158.0,Texas,".Aransas County, Texas"
5,.Archer County,8904.0,8849.0,8942.0,9013.0,9078.0,9068.0,9063.0,9026.0,9104.0,9023.0,9054.0,Texas,".Archer County, Texas"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3189,.Washington County,203014.0,203179.0,203104.0,203859.0,204090.0,205359.0,205992.0,206259.0,207240.0,207571.0,207820.0,Pennsylvania,".Washington County, Pennsylvania"
3190,.Wayne County,47679.0,47994.0,48330.0,48938.0,49423.0,50409.0,52031.0,52341.0,51692.0,52552.0,52822.0,Pennsylvania,".Wayne County, Pennsylvania"
3191,.Westmoreland County,369990.0,368698.0,367680.0,367836.0,367281.0,366946.0,366559.0,366318.0,366058.0,365259.0,365169.0,Pennsylvania,".Westmoreland County, Pennsylvania"
3192,.Wyoming County,28072.0,28119.0,28011.0,28074.0,28195.0,28160.0,28194.0,28287.0,28294.0,28268.0,28276.0,Pennsylvania,".Wyoming County, Pennsylvania"


In [12]:
combined_df['full_county_name'] = combined_df['full_county_name'].str.strip().str.lstrip('.').str.strip()

In [13]:
combined_df.drop('geographic area', axis=1, inplace=True)
combined_df.drop('state', axis=1, inplace=True)
combined_df

,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,full_county_name
1,55114.0,54263.0,54740.0,56068.0,56245.0,56873.0,57386.0,57870.0,57963.0,58410.0,58458.0,"Anderson County, Texas"
2,13002.0,12856.0,13022.0,12976.0,13006.0,13016.0,13195.0,13513.0,14099.0,14601.0,14786.0,"Andrews County, Texas"
3,80123.0,80273.0,80803.0,81510.0,82070.0,82553.0,83810.0,84518.0,84961.0,86029.0,86771.0,"Angelina County, Texas"
4,22457.0,22287.0,22616.0,22843.0,23067.0,23561.0,23395.0,23172.0,23225.0,23291.0,23158.0,"Aransas County, Texas"
5,8904.0,8849.0,8942.0,9013.0,9078.0,9068.0,9063.0,9026.0,9104.0,9023.0,9054.0,"Archer County, Texas"
...,...,...,...,...,...,...,...,...,...,...,...,...
3189,203014.0,203179.0,203104.0,203859.0,204090.0,205359.0,205992.0,206259.0,207240.0,207571.0,207820.0,"Washington County, Pennsylvania"
3190,47679.0,47994.0,48330.0,48938.0,49423.0,50409.0,52031.0,52341.0,51692.0,52552.0,52822.0,"Wayne County, Pennsylvania"
3191,369990.0,368698.0,367680.0,367836.0,367281.0,366946.0,366559.0,366318.0,366058.0,365259.0,365169.0,"Westmoreland County, Pennsylvania"
3192,28072.0,28119.0,28011.0,28074.0,28195.0,28160.0,28194.0,28287.0,28294.0,28268.0,28276.0,"Wyoming County, Pennsylvania"


In [14]:
combined_df

,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,full_county_name
1,55114.0,54263.0,54740.0,56068.0,56245.0,56873.0,57386.0,57870.0,57963.0,58410.0,58458.0,"Anderson County, Texas"
2,13002.0,12856.0,13022.0,12976.0,13006.0,13016.0,13195.0,13513.0,14099.0,14601.0,14786.0,"Andrews County, Texas"
3,80123.0,80273.0,80803.0,81510.0,82070.0,82553.0,83810.0,84518.0,84961.0,86029.0,86771.0,"Angelina County, Texas"
4,22457.0,22287.0,22616.0,22843.0,23067.0,23561.0,23395.0,23172.0,23225.0,23291.0,23158.0,"Aransas County, Texas"
5,8904.0,8849.0,8942.0,9013.0,9078.0,9068.0,9063.0,9026.0,9104.0,9023.0,9054.0,"Archer County, Texas"
...,...,...,...,...,...,...,...,...,...,...,...,...
3189,203014.0,203179.0,203104.0,203859.0,204090.0,205359.0,205992.0,206259.0,207240.0,207571.0,207820.0,"Washington County, Pennsylvania"
3190,47679.0,47994.0,48330.0,48938.0,49423.0,50409.0,52031.0,52341.0,51692.0,52552.0,52822.0,"Wayne County, Pennsylvania"
3191,369990.0,368698.0,367680.0,367836.0,367281.0,366946.0,366559.0,366318.0,366058.0,365259.0,365169.0,"Westmoreland County, Pennsylvania"
3192,28072.0,28119.0,28011.0,28074.0,28195.0,28160.0,28194.0,28287.0,28294.0,28268.0,28276.0,"Wyoming County, Pennsylvania"


In [15]:
# Split on the last comma (just in case some counties have commas)
combined_df[['County', 'State']] = combined_df['full_county_name'].str.rsplit(',', n=1, expand=True)

# Strip whitespace from the new columns
combined_df['County'] = combined_df['County'].str.strip()
combined_df['State'] = combined_df['State'].str.strip()

Rearrange column order and give it a new name.

In [16]:
population_2000_2010 = combined_df[['State', 'County', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010']]

In [17]:
population_2000_2010 = population_2000_2010.sort_values(by=['State', 'County'])
population_2000_2010

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010
1191,Alabama,Autauga County,43751.0,44889.0,45909.0,46800.0,48366.0,49676.0,51328.0,52405.0,53277.0,54135.0,54571.0
1192,Alabama,Baldwin County,140416.0,144875.0,147957.0,151509.0,156266.0,162183.0,168121.0,172404.0,175827.0,179406.0,182265.0
1193,Alabama,Barbour County,29042.0,28863.0,28653.0,28594.0,28287.0,28027.0,27861.0,27757.0,27808.0,27657.0,27457.0
1194,Alabama,Bibb County,19856.0,21028.0,21199.0,21399.0,21721.0,22042.0,22099.0,22438.0,22705.0,22941.0,22915.0
1195,Alabama,Blount County,50982.0,51845.0,52551.0,53457.0,54124.0,54624.0,55485.0,56240.0,57055.0,57341.0,57322.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3121,Wyoming,Sweetwater County,37620.0,36899.0,37428.0,37450.0,38026.0,38739.0,39749.0,41470.0,42358.0,44133.0,43806.0
3122,Wyoming,Teton County,18257.0,18653.0,18837.0,19066.0,19467.0,19632.0,20014.0,20472.0,20988.0,21232.0,21294.0
3123,Wyoming,Uinta County,19745.0,19413.0,19587.0,19480.0,19470.0,19494.0,19709.0,20171.0,20613.0,21054.0,21118.0
3124,Wyoming,Washakie County,8292.0,8068.0,7988.0,7976.0,7960.0,8022.0,7979.0,8169.0,8229.0,8423.0,8533.0


# 2011 to 2020

This part was much easier, as all data was contained in a single Excel file. I did some basic pre-processing in the word processor RowZero and downloaded it as CSV which is loaded below

In [18]:
population_2011_2020 = pd.read_csv('/Users/matthewcavanaugh/Desktop/GitHub/us-presidential-elections-by-county/sources/population raw/State Pop Estimates 2011 to 2020 raw.csv')

In [19]:
population_2011_2020

,full_county_name,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
0,".Autauga County, Alabama",55564,55574,55619,56062,56311,56979,57393,57747,58251,58814
1,".Baldwin County, Alabama",187051,191053,196205,200910,205083,210148,215475,221187,227057,231766
2,".Barbour County, Alabama",27416,27301,27132,27012,26600,26186,25585,25359,25187,25225
3,".Bibb County, Alabama",22749,22680,22543,22585,22607,22654,22607,22385,22408,22302
4,".Blount County, Alabama",57711,57855,58020,58045,58165,58237,58671,58760,58949,59132
...,...,...,...,...,...,...,...,...,...,...,...
3138,".Sweetwater County, Wyoming",43944,44931,45044,44806,44546,44040,43339,42820,42504,42271
3139,".Teton County, Wyoming",21406,21614,22293,22747,23015,23174,23289,23155,23266,23324
3140,".Uinta County, Wyoming",20936,21071,21059,20953,20923,20885,20651,20528,20453,20447
3141,".Washakie County, Wyoming",8443,8394,8393,8246,8244,8135,7960,7825,7757,7683


In [20]:
population_2011_2020['full_county_name'] = population_2011_2020['full_county_name'].str.strip().str.lstrip('.').str.strip()

# Split on the last comma (just in case some counties have commas)
population_2011_2020[['County', 'State']] = population_2011_2020['full_county_name'].str.rsplit(',', n=1, expand=True)

In [21]:
population_2011_2020.drop('full_county_name', axis=1, inplace=True)
population_2011_2020 = population_2011_2020[['State', 'County', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020']]
population_2011_2020

,State,County,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
0,Alabama,Autauga County,55564,55574,55619,56062,56311,56979,57393,57747,58251,58814
1,Alabama,Baldwin County,187051,191053,196205,200910,205083,210148,215475,221187,227057,231766
2,Alabama,Barbour County,27416,27301,27132,27012,26600,26186,25585,25359,25187,25225
3,Alabama,Bibb County,22749,22680,22543,22585,22607,22654,22607,22385,22408,22302
4,Alabama,Blount County,57711,57855,58020,58045,58165,58237,58671,58760,58949,59132
...,...,...,...,...,...,...,...,...,...,...,...,...
3138,Wyoming,Sweetwater County,43944,44931,45044,44806,44546,44040,43339,42820,42504,42271
3139,Wyoming,Teton County,21406,21614,22293,22747,23015,23174,23289,23155,23266,23324
3140,Wyoming,Uinta County,20936,21071,21059,20953,20923,20885,20651,20528,20453,20447
3141,Wyoming,Washakie County,8443,8394,8393,8246,8244,8135,7960,7825,7757,7683


# Combining

In [22]:
keys_2000_2010 = set(population_2000_2010[['State', 'County']].apply(tuple, axis=1))
keys_2011_2020 = set(population_2011_2020[['State', 'County']].apply(tuple, axis=1))

In [23]:
only_in_2000_2010 = keys_2000_2010 - keys_2011_2020
only_in_2011_2020 = keys_2011_2020 - keys_2000_2010

Concatenation was failing because population_2011_2020['State'] has leading blank spaces for every value. Let's strip them.

In [24]:
population_2011_2020['State'] = population_2011_2020['State'].str.strip()

In [25]:
keys_2000_2010 = set(population_2000_2010[['State', 'County']].apply(tuple, axis=1))
keys_2011_2020 = set(population_2011_2020[['State', 'County']].apply(tuple, axis=1))

only_in_2011_2020 = keys_2011_2020 - keys_2000_2010

print("\nEntries in 2011–2020 but NOT in 2000–2010:")
for state_county in sorted(only_in_2011_2020):
    print(repr(state_county))  # repr shows spaces clearly


Entries in 2011–2020 but NOT in 2000–2010:
('Alaska', 'Chugach Census Area')
('Alaska', 'Copper River Census Area')
('Alaska', 'Kusilvak Census Area')
('Alaska', 'Petersburg Borough')
('Louisiana', 'LaSalle Parish')
('South Dakota', 'Oglala Lakota County')


LaSalle Parish, Louisiana, was spelled with a space ("La Salle") in the 2000-2010 dataset, and without one in 2011-2020. We will remove the space in the former.

In [26]:
population_2000_2010[population_2000_2010['County'].str.contains('La Salle')]

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010
1879,Louisiana,La Salle Parish,14276.0,14207.0,14359.0,14356.0,14366.0,14313.0,14519.0,14570.0,14667.0,14717.0,14890.0
142,Texas,La Salle County,5863.0,5934.0,6098.0,6169.0,6322.0,6542.0,6549.0,6675.0,6653.0,6771.0,6886.0


In [27]:
mask = (
    (population_2000_2010['State'] == 'Louisiana') &
    (population_2000_2010['County'] == 'La Salle Parish')
)

population_2000_2010.loc[mask, 'County'] = 'LaSalle Parish'

In [28]:
population_2000_2010[population_2000_2010['County'].str.contains('La Salle')]

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010
142,Texas,La Salle County,5863.0,5934.0,6098.0,6169.0,6322.0,6542.0,6549.0,6675.0,6653.0,6771.0,6886.0


In [29]:
population_2000_2010[population_2000_2010['County'].str.contains('LaSalle')]

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010
863,Illinois,LaSalle County,111599.0,112212.0,111904.0,110949.0,112135.0,112565.0,113139.0,113794.0,114171.0,113999.0,113924.0
1879,Louisiana,LaSalle Parish,14276.0,14207.0,14359.0,14356.0,14366.0,14313.0,14519.0,14570.0,14667.0,14717.0,14890.0


La Salle County, Texas, is supposed to have a space.

Shannon / Oglala Lakota County, South Dakota, is a problem we've run into before. It changed its name from the former to the latter in 2015.

In [30]:
population_2000_2010[population_2000_2010['County'].str.contains('Shannon')]

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010
1175,Missouri,Shannon County,8319.0,8303.0,8273.0,8213.0,8254.0,8300.0,8313.0,8402.0,8420.0,8446.0,8441.0
2644,South Dakota,Shannon County,12466.0,12602.0,12872.0,12993.0,12983.0,13150.0,13404.0,13345.0,13368.0,13425.0,13586.0


In [31]:
population_2011_2020[population_2011_2020['County'].str.contains('Shannon')]

,State,County,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
1584,Missouri,Shannon County,8268,8070,7910,7799,7662,7495,7392,7229,7097,7028


This dataset seems to have used the former name until 2010 and the latter from 2011 onward. We can update the older one to the new name.

In [32]:
mask = (
    (population_2000_2010['State'] == 'South Dakota') &
    (population_2000_2010['County'] == 'Shannon County')
)

population_2000_2010.loc[mask, 'County'] = 'Oglala Lakota County'

In [33]:
population_2000_2010[population_2000_2010['County'].str.contains('Shannon')]

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010
1175,Missouri,Shannon County,8319.0,8303.0,8273.0,8213.0,8254.0,8300.0,8313.0,8402.0,8420.0,8446.0,8441.0


In [34]:
population_2000_2010[population_2000_2010['County'].str.contains('Oglala')]

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010
2644,South Dakota,Oglala Lakota County,12466.0,12602.0,12872.0,12993.0,12983.0,13150.0,13404.0,13345.0,13368.0,13425.0,13586.0


Finally, we will have to drop Alaska as we have done before.

In [35]:
population_2000_2010 = population_2000_2010[~population_2000_2010['State'].str.contains('Alaska', na=False)]
population_2011_2020 = population_2011_2020[~population_2011_2020['State'].str.contains('Alaska', na=False)]

In [36]:
keys_2000_2010 = set(population_2000_2010[['State', 'County']].apply(tuple, axis=1))
keys_2011_2020 = set(population_2011_2020[['State', 'County']].apply(tuple, axis=1))

only_in_2011_2020 = keys_2011_2020 - keys_2000_2010

print("\nEntries in 2011–2020 but NOT in 2000–2010:")
for state_county in sorted(only_in_2011_2020):
    print(repr(state_county))  # repr shows spaces clearly


Entries in 2011–2020 but NOT in 2000–2010:


In [37]:
population_2000_2010[population_2000_2010['State'].str.contains('Alaska')]

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010


In [38]:
population_2011_2020[population_2011_2020['State'].str.contains('Alaska')]

,State,County,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020


In [39]:
population_2000_2010.shape

(3114, 13)

In [40]:
population_2011_2020.shape

(3113, 12)

The 2000-2010 dataset still has one more row than 2011-2020. I think I know what it is.

In [41]:
population_2000_2010[population_2000_2010['County'].str.contains('Bedford')]

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010
3131,Pennsylvania,Bedford County,49890.0,49778.0,49742.0,49812.0,49829.0,49691.0,49871.0,49889.0,50052.0,49889.0,49762.0
2494,Tennessee,Bedford County,37608.0,38414.0,38960.0,39697.0,40558.0,41616.0,42780.0,43678.0,44395.0,44845.0,45058.0
2686,Virginia,Bedford County,60226.0,60985.0,61580.0,62441.0,63563.0,64825.0,66363.0,67392.0,67594.0,68399.0,68676.0
2773,Virginia,Bedford city,6421.0,6400.0,6320.0,6237.0,6198.0,6186.0,6147.0,6169.0,6150.0,6174.0,6222.0


In [42]:
population_2011_2020[population_2011_2020['County'].str.contains('Bedford')]

,State,County,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
2249,Pennsylvania,Bedford County,49587,49523,49217,49000,48586,48398,48135,47835,47627,47571
2430,Tennessee,Bedford County,45261,45277,45598,46298,47005,47515,48285,49233,49774,50234
2830,Virginia,Bedford County,75513,75679,76081,76912,77477,77815,78202,78839,79121,79465


Just as I thought. Bedford city gave up independent city in 2013 and merged into Bedford County. This is a problem I've run into before.

In [43]:
population_2000_2010 = population_2000_2010[~population_2000_2010['County'].str.contains('Bedford city', na=False)]

In [44]:
population_2000_2010[population_2000_2010['County'].str.contains('Bedford')]

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010
3131,Pennsylvania,Bedford County,49890.0,49778.0,49742.0,49812.0,49829.0,49691.0,49871.0,49889.0,50052.0,49889.0,49762.0
2494,Tennessee,Bedford County,37608.0,38414.0,38960.0,39697.0,40558.0,41616.0,42780.0,43678.0,44395.0,44845.0,45058.0
2686,Virginia,Bedford County,60226.0,60985.0,61580.0,62441.0,63563.0,64825.0,66363.0,67392.0,67594.0,68399.0,68676.0


In [45]:
population_2000_2010.shape

(3113, 13)

In [46]:
population_2011_2020.shape

(3113, 12)

The uneven number of columns is to be expected, considering the different number of years covered.

In [47]:
for df in [population_2000_2010, population_2011_2020]:
    df['State'] = df['State'].str.strip()
    df['County'] = df['County'].str.strip()

/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_4111/1689012998.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['State'] = df['State'].str.strip()
/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_4111/1689012998.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['County'] = df['County'].str.strip()


In [48]:
combined_population = pd.merge(
    population_2000_2010,
    population_2011_2020,
    on=['State', 'County'],
    how='inner'
)

In [49]:
combined_population

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,...,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
0,Alabama,Autauga County,43751.0,44889.0,45909.0,46800.0,48366.0,49676.0,51328.0,52405.0,...,55564,55574,55619,56062,56311,56979,57393,57747,58251,58814
1,Alabama,Baldwin County,140416.0,144875.0,147957.0,151509.0,156266.0,162183.0,168121.0,172404.0,...,187051,191053,196205,200910,205083,210148,215475,221187,227057,231766
2,Alabama,Barbour County,29042.0,28863.0,28653.0,28594.0,28287.0,28027.0,27861.0,27757.0,...,27416,27301,27132,27012,26600,26186,25585,25359,25187,25225
3,Alabama,Bibb County,19856.0,21028.0,21199.0,21399.0,21721.0,22042.0,22099.0,22438.0,...,22749,22680,22543,22585,22607,22654,22607,22385,22408,22302
4,Alabama,Blount County,50982.0,51845.0,52551.0,53457.0,54124.0,54624.0,55485.0,56240.0,...,57711,57855,58020,58045,58165,58237,58671,58760,58949,59132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3108,Wyoming,Sweetwater County,37620.0,36899.0,37428.0,37450.0,38026.0,38739.0,39749.0,41470.0,...,43944,44931,45044,44806,44546,44040,43339,42820,42504,42271
3109,Wyoming,Teton County,18257.0,18653.0,18837.0,19066.0,19467.0,19632.0,20014.0,20472.0,...,21406,21614,22293,22747,23015,23174,23289,23155,23266,23324
3110,Wyoming,Uinta County,19745.0,19413.0,19587.0,19480.0,19470.0,19494.0,19709.0,20171.0,...,20936,21071,21059,20953,20923,20885,20651,20528,20453,20447
3111,Wyoming,Washakie County,8292.0,8068.0,7988.0,7976.0,7960.0,8022.0,7979.0,8169.0,...,8443,8394,8393,8246,8244,8135,7960,7825,7757,7683


In [50]:
# Identify year columns (assuming all year columns are digits)
year_cols = [col for col in combined_population.columns if col.isdigit()]

# Convert those columns to integers, assuming no NaNs
combined_population[year_cols] = combined_population[year_cols].astype(int)

In [51]:
combined_population

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,...,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
0,Alabama,Autauga County,43751,44889,45909,46800,48366,49676,51328,52405,...,55564,55574,55619,56062,56311,56979,57393,57747,58251,58814
1,Alabama,Baldwin County,140416,144875,147957,151509,156266,162183,168121,172404,...,187051,191053,196205,200910,205083,210148,215475,221187,227057,231766
2,Alabama,Barbour County,29042,28863,28653,28594,28287,28027,27861,27757,...,27416,27301,27132,27012,26600,26186,25585,25359,25187,25225
3,Alabama,Bibb County,19856,21028,21199,21399,21721,22042,22099,22438,...,22749,22680,22543,22585,22607,22654,22607,22385,22408,22302
4,Alabama,Blount County,50982,51845,52551,53457,54124,54624,55485,56240,...,57711,57855,58020,58045,58165,58237,58671,58760,58949,59132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3108,Wyoming,Sweetwater County,37620,36899,37428,37450,38026,38739,39749,41470,...,43944,44931,45044,44806,44546,44040,43339,42820,42504,42271
3109,Wyoming,Teton County,18257,18653,18837,19066,19467,19632,20014,20472,...,21406,21614,22293,22747,23015,23174,23289,23155,23266,23324
3110,Wyoming,Uinta County,19745,19413,19587,19480,19470,19494,19709,20171,...,20936,21071,21059,20953,20923,20885,20651,20528,20453,20447
3111,Wyoming,Washakie County,8292,8068,7988,7976,7960,8022,7979,8169,...,8443,8394,8393,8246,8244,8135,7960,7825,7757,7683


The newly combined dataframe still has two more rows than my primary dataest. I think I know what they are.

In [52]:
combined_population[combined_population['County'].str.contains('Broomfield')]

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,...,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
222,Colorado,Broomfield County,39202,41055,41800,43805,44867,46313,48729,51526,...,57763,59473,60884,62739,66085,67991,69958,71272,72681,74110


In [53]:
combined_population[combined_population['County'].str.contains('Kalawao')]

,State,County,2000,2001,2002,2003,2004,2005,2006,2007,...,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
519,Hawaii,Kalawao County,147,138,133,127,117,114,109,105,...,89,87,86,85,83,82,81,79,79,81


In [54]:
combined_population = combined_population[~combined_population['County'].str.contains('Broomfield County', na=False)]
combined_population = combined_population[~combined_population['County'].str.contains('Kalawao Count', na=False)]

In [55]:
combined_population.shape

(3111, 23)

In [56]:
population_pivoted = combined_population.melt(
    id_vars=['State', 'County'],
    var_name='Year',
    value_name='Population'
)

In [57]:
population_pivoted

,State,County,Year,Population
0,Alabama,Autauga County,2000,43751
1,Alabama,Baldwin County,2000,140416
2,Alabama,Barbour County,2000,29042
3,Alabama,Bibb County,2000,19856
4,Alabama,Blount County,2000,50982
...,...,...,...,...
65326,Wyoming,Sweetwater County,2020,42271
65327,Wyoming,Teton County,2020,23324
65328,Wyoming,Uinta County,2020,20447
65329,Wyoming,Washakie County,2020,7683


In [58]:
population_pivoted['Year'] = population_pivoted['Year'].astype(int)

In [59]:
years_to_keep = [2000, 2004, 2008, 2012, 2016, 2020]
population_pivoted = population_pivoted[population_pivoted['Year'].isin(years_to_keep)]

In [60]:
population_pivoted = population_pivoted[['Year', 'State', 'County', 'Population']]
population_pivoted

,Year,State,County,Population
0,2000,Alabama,Autauga County,43751
1,2000,Alabama,Baldwin County,140416
2,2000,Alabama,Barbour County,29042
3,2000,Alabama,Bibb County,19856
4,2000,Alabama,Blount County,50982
...,...,...,...,...
65326,2020,Wyoming,Sweetwater County,42271
65327,2020,Wyoming,Teton County,23324
65328,2020,Wyoming,Uinta County,20447
65329,2020,Wyoming,Washakie County,7683


In [61]:
# Clean up text for aesthetic purposes
def clean_text(text):
    if isinstance(text, str):
        return text.title()  # Capitalizes the first letter of each word and makes the rest lowercase
    return text  # Return text unchanged if it's not a string
population_pivoted['County'] = population_pivoted['County'].apply(clean_text)

In [62]:
population_pivoted[population_pivoted['County'].str.contains('City')]

,Year,State,County,Population
1165,2000,Maryland,Baltimore City,651262
1553,2000,Missouri,St. Louis City,348192
1716,2000,Nevada,Carson City,52477
2791,2000,Virginia,Alexandria City,128278
2802,2000,Virginia,Bristol City,17346
...,...,...,...,...
65128,2020,Virginia,Suffolk City,94325
65132,2020,Virginia,Virginia Beach City,459471
65135,2020,Virginia,Waynesboro City,22195
65137,2020,Virginia,Williamsburg City,15424


# Spelling

On a previous run through of merging, two counties were lost likely due to spelling mismatches. I'll intercept the problem and change them manually here.

In [63]:
stmarys_pop = population_pivoted[population_pivoted['State'].str.contains('Maryland')]
stmarys_pop = stmarys_pop[stmarys_pop['County'].str.contains('Mary')]
stmarys_pop

,Year,State,County,Population
1181,2000,Maryland,St. Mary'S County,86196
13625,2004,Maryland,St. Mary'S County,94900
26069,2008,Maryland,St. Mary'S County,101921
38513,2012,Maryland,St. Mary'S County,108688
50957,2016,Maryland,St. Mary'S County,111444
63401,2020,Maryland,St. Mary'S County,113781


In [64]:
population_pivoted.loc[
    (population_pivoted['State'] == 'Maryland') & 
    (population_pivoted['County'] == "St. Mary'S County"), 
    'County'
] = "St. Mary's County"

In [65]:
stmarys_pop = population_pivoted[population_pivoted['State'].str.contains('Maryland')]
stmarys_pop = stmarys_pop[stmarys_pop['County'].str.contains('Mary')]
stmarys_pop

,Year,State,County,Population
1181,2000,Maryland,St. Mary's County,86196
13625,2004,Maryland,St. Mary's County,94900
26069,2008,Maryland,St. Mary's County,101921
38513,2012,Maryland,St. Mary's County,108688
50957,2016,Maryland,St. Mary's County,111444
63401,2020,Maryland,St. Mary's County,113781


In [67]:
dc_pop = population_pivoted[population_pivoted['County'].str.contains('District')]
dc_pop

,Year,State,County,Population
289,2000,District of Columbia,District Of Columbia,572086
12733,2004,District of Columbia,District Of Columbia,567754
25177,2008,District of Columbia,District Of Columbia,580236
37621,2012,District of Columbia,District Of Columbia,630643
50065,2016,District of Columbia,District Of Columbia,673428
62509,2020,District of Columbia,District Of Columbia,689548


In [68]:
population_pivoted.loc[
    (population_pivoted['State'] == 'D') & 
    (population_pivoted['County'] == "District Of Columbia"), 
    'State'
] = "District Of Columbia"

In [69]:
dc_pop = population_pivoted[population_pivoted['County'].str.contains('District')]
dc_pop

,Year,State,County,Population
289,2000,District Of Columbia,District Of Columbia,572086
12733,2004,District Of Columbia,District Of Columbia,567754
25177,2008,District Of Columbia,District Of Columbia,580236
37621,2012,District Of Columbia,District Of Columbia,630643
50065,2016,District Of Columbia,District Of Columbia,673428
62509,2020,District Of Columbia,District Of Columbia,689548


In [70]:
# Specify the desired file path (e.g., in a 'data' subfolder)
file_path = '/Users/matthewcavanaugh/Desktop/GitHub/us-presidential-elections-by-county/sources/Population CLEANED.csv'

# Save the DataFrame to CSV at the specified path, without including the index
population_pivoted.to_csv(file_path, index=False)